In [1]:
#Creating a clean master directory for the unzipped files
!mkdir -p /content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped
#Extracting all 5 datasets
!unzip -o -q "/content/drive/MyDrive/Capstone_Project/capstone_datasets/skeletons/asdpose_preprocessed.zip" -d "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/asdpose"
!unzip -o -q "/content/drive/MyDrive/Capstone_Project/capstone_datasets/skeletons/hbd21_test_train_split.zip" -d "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/hbd21"
!unzip -o -q "/content/drive/MyDrive/Capstone_Project/capstone_datasets/skeletons/le2i_preprocessed.zip" -d "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/le2i"
!unzip -o -q "/content/drive/MyDrive/Capstone_Project/capstone_datasets/skeletons/ssbd_model_ready.zip" -d "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/ssbd"
!unzip -o -q "/content/drive/MyDrive/Capstone_Project/capstone_datasets/skeletons/ur_fall_processed.zip" -d "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/ur_fall"
print("All 5 datasets extracted successfully!")

All 5 datasets extracted successfully!


In [ ]:
#Checking class distribution
import os
import glob
from collections import Counter
WORKING_DIR = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped"

dataset_inventory = []
# Finding all .npy files across all 5 extracted folders
all_npy_files = glob.glob(os.path.join(WORKING_DIR, "**", "*.npy"), recursive=True)
for file_path in all_npy_files:
    path_lower = file_path.lower()
    label = "Unknown"
    #Class 1: Stimming / SMM (ASDPose & SSBD)
    if "asdpose" in path_lower or "ssbd" in path_lower:
        label = "Class 1: Stimming"
    #Class 0 & 2: Falls and ADL (UR Fall & Le2i)
    elif "ur_fall" in path_lower or "le2i" in path_lower:
        if "not fall" in path_lower or "adl" in path_lower or "normal" in path_lower:
            label = "Class 0: Normal ADL"
        elif "fall" in path_lower:
            label = "Class 2: Fall"
        else:
            label = "Class 2: Fall (Check Filename)"

    #Class 3 & 0: Aggression and Normal (HBD21)
    elif "hbd21" in path_lower:
        if "normal" in path_lower or "adl" in path_lower:
            label = "Class 0: Normal ADL"
        else:
            label = "Class 3: Aggression / Anomaly"
    # Extracting the top-level dataset folder name for the report
    dataset_source = file_path.replace(WORKING_DIR, "").strip("/").split('/')[0]
    dataset_inventory.append({
        "file": file_path,
        "dataset_source": dataset_source,
        "class_label": label
    })
#Balance Report
print(f"Total .npy files found: {len(dataset_inventory)}\n")
print("Data Balance by Class")
category_counts = Counter([item["class_label"] for item in dataset_inventory])
for cat, count in category_counts.most_common():
    print(f" - {cat}: {count} samples")
print("\nData Contribution by Dataset")
source_counts = Counter([item["dataset_source"] for item in dataset_inventory])
for src, count in source_counts.most_common():
    print(f" - {src}: {count} samples")

In [ ]:
import numpy as np

#Inspecting HBD21
HBD_X_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/hbd21/hbd21_windows/X_train.npy"
HBD_Y_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/hbd21/hbd21_windows/y_train.npy"

try:
    hbd_x = np.load(HBD_X_PATH)
    hbd_y = np.load(HBD_Y_PATH)
    print("--- HBD21 Dataset ---")
    print(f"X_train shape (The skeleton data): {hbd_x.shape}")
    print(f"y_train shape (The labels): {hbd_y.shape}")
    print(f"Unique class labels in y_train: {np.unique(hbd_y)}")
except Exception as e:
    print(f"Error loading HBD21: {e}")

# 2. Inspecting Le2i Annotations
LE2I_ANN_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped/le2i/le2i_master_annotations.npy"

try:
    le2i_ann = np.load(LE2I_ANN_PATH, allow_pickle=True)
    print("\n--- Le2i Annotations ---")
    print(f"Array shape: {le2i_ann.shape}")
    print("First 3 annotation entries:")
    for i in range(min(3, len(le2i_ann))):
        print(le2i_ann[i])
except Exception as e:
    print(f"Error loading Le2i annotations: {e}")

--- HBD21 Dataset ---
X_train shape (The skeleton data): (5345, 30, 33, 3)
y_train shape (The labels): (5345,)
Unique class labels in y_train: [0 1 2 3]

--- Le2i Annotations ---
Array shape: (180,)
First 3 annotation entries:
{'folder': 'Coffee_room_01', 'video_id': 1, 'label': 1, 'start_frame': 48, 'end_frame': 80, 'annotation_file': 'video (1).txt'}
{'folder': 'Coffee_room_01', 'video_id': 2, 'label': 1, 'start_frame': 192, 'end_frame': 219, 'annotation_file': 'video (2).txt'}
{'folder': 'Coffee_room_01', 'video_id': 3, 'label': 1, 'start_frame': 223, 'end_frame': 261, 'annotation_file': 'video (3).txt'}


In [ ]:
import os
import json
import glob
import random
import numpy as np
from tqdm import tqdm

WORKING_DIR = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped"
OUTPUT_DIR  = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "hbd21_unpacked"), exist_ok=True)

class_0_normal = []
class_1_stimming = []
class_2_fall = []
class_3_aggression = []

#Unpacking HBD21 (Aggression & Normal) ---
print("Unpacking HBD21 arrays...")
hbd_x = np.load(os.path.join(WORKING_DIR, "hbd21/hbd21_windows/X_train.npy"))
hbd_y = np.load(os.path.join(WORKING_DIR, "hbd21/hbd21_windows/y_train.npy"))

for i in tqdm(range(len(hbd_x))):
    # HBD21 Label mapping (Assuming 0=Normal, 1/2/3=Anomalies/Aggression based on typical sets)
    label = 0 if hbd_y[i] == 0 else 3
    file_path = os.path.join(OUTPUT_DIR, f"hbd21_unpacked/seq_{i}.npy")

    # Saving individual slice if it doesn't exist to save time
    if not os.path.exists(file_path):
        np.save(file_path, hbd_x[i])

    if label == 0:
        class_0_normal.append({"file_path": file_path, "label": 0, "category": "Normal ADL"})
    else:
        class_3_aggression.append({"file_path": file_path, "label": 3, "category": "Aggression"})

# --- 2. Mapoing Le2i Annotations (Falls) ---
print("Mapping Le2i annotations...")
le2i_ann = np.load(os.path.join(WORKING_DIR, "le2i/le2i_master_annotations.npy"), allow_pickle=True)
for entry in le2i_ann:
    # Matching the annotation dict to the physical .npy file
    folder = entry['folder']
    vid_id = entry['video_id']
    target_file = f"{folder}---video ({vid_id}).npy"

    # Searching for this specific file in the Le2i directory
    file_match = glob.glob(os.path.join(WORKING_DIR, f"le2i/**/{target_file}"), recursive=True)
    if file_match:
        # Assuming label 1 in Le2i means Fall, 0 means Normal
        assigned_label = 2 if entry['label'] == 1 else 0
        cat_name = "Fall" if assigned_label == 2 else "Normal ADL"

        data_dict = {"file_path": file_match[0], "label": assigned_label, "category": cat_name}
        if assigned_label == 2:
            class_2_fall.append(data_dict)
        else:
            class_0_normal.append(data_dict)

#Processing ASDPose & SSBD (Stimming) ---
print("Gathering Stimming data...")
for f in glob.glob(os.path.join(WORKING_DIR, "asdpose/**/*.npy"), recursive=True):
    class_1_stimming.append({"file_path": f, "label": 1, "category": "Stimming"})

#Processing UR Fall ---
print("Gathering UR Fall data...")
for f in glob.glob(os.path.join(WORKING_DIR, "ur_fall/**/*.npy"), recursive=True):
    if "not fall" in f.lower() or "adl" in f.lower():
        class_0_normal.append({"file_path": f, "label": 0, "category": "Normal ADL"})
    elif "fall" in f.lower():
        class_2_fall.append({"file_path": f, "label": 2, "category": "Fall"})

#Balancing the Datasets ---
MIN_SAMPLES = min(len(class_0_normal), len(class_1_stimming), len(class_2_fall), len(class_3_aggression))
print(f"\nBalancing all classes to exactly {MIN_SAMPLES} samples...")

balanced_master = (
    random.sample(class_0_normal, MIN_SAMPLES) +
    random.sample(class_1_stimming, MIN_SAMPLES) +
    random.sample(class_2_fall, MIN_SAMPLES) +
    random.sample(class_3_aggression, MIN_SAMPLES)
)
random.shuffle(balanced_master)

BALANCED_INDEX_PATH = os.path.join(OUTPUT_DIR, "master_balanced_4class_index.json")
with open(BALANCED_INDEX_PATH, "w") as f:
    json.dump(balanced_master, f, indent=4)

print(f"Success! Master index saved to {BALANCED_INDEX_PATH}")

Unpacking HBD21 arrays...


100%|██████████| 5345/5345 [01:32<00:00, 57.65it/s]


Mapping Le2i annotations...
Gathering Stimming data...
Gathering UR Fall data...

Balancing all classes to exactly 117 samples...
Success! Master index saved to /content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset/master_balanced_4class_index.json


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#UNIVERSAL SKELETON LOADER (Handles 2D, 3D, 4D arrays) ---
class StandardizedSkeletonDataset(Dataset):
    def __init__(self, metadata, target_frames=60, target_joints=33):
        self.metadata = metadata
        self.T = target_frames
        self.V = target_joints

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        item = self.metadata[idx]
        data = np.load(item["file_path"])
        data = np.array(data, dtype=np.float32)

        # 1. Handling 2D Arrays e.g., (Frames, Joints*Channels)
        if data.ndim == 2:
            t_len, feat_len = data.shape
            if feat_len % 2 == 0 and (feat_len // 2) <= self.V:
                v_len, c_len = feat_len // 2, 2
                data = data.reshape(t_len, v_len, c_len)
            elif feat_len % 3 == 0 and (feat_len // 3) <= self.V:
                v_len, c_len = feat_len // 3, 3
                data = data.reshape(t_len, v_len, c_len)

        # 2. Handling 3D Arrays -> Convert to (Channels, Frames, Joints)
        if data.ndim == 3:
            # Case A: (Frames, Joints, Channels) -> e.g., (30, 33, 3) or (750, 17, 2)
            if data.shape[-1] in [2, 3]:
                data = np.transpose(data, (2, 0, 1))
            # Case B: (Frames, Channels, Joints)
            elif data.shape[1] in [2, 3]:
                data = np.transpose(data, (1, 0, 2))
            # Case C: Already (Channels, Frames, Joints) -> (3, 150, 17)
            data = np.expand_dims(data, axis=-1)  # Add person dimension (C, T, V, 1)

        # 3. Handling 4D Arrays -> Ensure (Channels, Frames, Joints, Persons)
        elif data.ndim == 4:
            if data.shape[0] not in [2, 3] and data.shape[-1] in [2, 3]:
                data = np.transpose(data, (3, 0, 1, 2))

        # Padding missing dimensions if still < 4D
        while data.ndim < 4:
            data = np.expand_dims(data, axis=-1)

        C_curr, T_curr, V_curr, _ = data.shape

        # Building blank target zero-tensor (3 Channels, target_frames, target_joints, 1 Person)
        out_data = np.zeros((3, self.T, self.V, 1), dtype=np.float32)

        c_limit = min(C_curr, 3)
        t_limit = min(T_curr, self.T)
        v_limit = min(V_curr, self.V)

        out_data[:c_limit, :t_limit, :v_limit, 0] = data[:c_limit, :t_limit, :v_limit, 0]

        tensor_data = torch.from_numpy(out_data).float()
        label = torch.tensor(item["label"], dtype=torch.long)
        return tensor_data, label


#Loading Metadata and Split Data ---
INDEX_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset/master_balanced_4class_index.json"
with open(INDEX_PATH, "r") as f:
    balanced_meta = json.load(f)

labels = [x["label"] for x in balanced_meta]
train_meta, val_meta = train_test_split(balanced_meta, test_size=0.2, random_state=42, stratify=labels)

train_loader = DataLoader(StandardizedSkeletonDataset(train_meta), batch_size=16, shuffle=True)
val_loader = DataLoader(StandardizedSkeletonDataset(val_meta), batch_size=16, shuffle=False)


#ST-GCN Architecture (4-Class) ---
class BaselineSTGCN_4Class(nn.Module):
    def __init__(self, num_classes=4):
        super(BaselineSTGCN_4Class, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=(5, 1), padding=(2, 0))
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(32, 64, kernel_size=(5, 1), stride=(2, 1), padding=(2, 0))
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.view(N, C, T, V * M)  # Flatten persons into spatial dim

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = x.view(N, -1)
        x = self.dropout(x)
        return self.fc(x)


#Training Loop
model = BaselineSTGCN_4Class(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
best_val_acc = 0.0
MODEL_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset/baseline_stgcn_4class.pth"

print(f"Starting 4-Class Training on {device}...")
print(f"Train samples: {len(train_meta)} | Val samples: {len(val_meta)}")
print("-" * 60)

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

print("-" * 60)
print(f"Baseline Training Complete! Best Val Accuracy: {best_val_acc:.1f}%")
print(f"Saved model to: {MODEL_PATH}")

Starting 4-Class Training on cuda...
Train samples: 374 | Val samples: 94
------------------------------------------------------------
Epoch [01/20] | Loss: 1.1547 | Train Acc: 44.9% | Val Acc: 45.7%
Epoch [02/20] | Loss: 1.0511 | Train Acc: 45.7% | Val Acc: 44.7%
Epoch [03/20] | Loss: 1.0337 | Train Acc: 43.3% | Val Acc: 44.7%
Epoch [04/20] | Loss: 1.0164 | Train Acc: 48.4% | Val Acc: 44.7%
Epoch [05/20] | Loss: 1.0053 | Train Acc: 49.5% | Val Acc: 44.7%
Epoch [06/20] | Loss: 0.9898 | Train Acc: 56.7% | Val Acc: 44.7%
Epoch [07/20] | Loss: 0.9678 | Train Acc: 55.3% | Val Acc: 46.8%
Epoch [08/20] | Loss: 0.9378 | Train Acc: 60.4% | Val Acc: 53.2%
Epoch [09/20] | Loss: 0.9070 | Train Acc: 60.7% | Val Acc: 48.9%
Epoch [10/20] | Loss: 0.8958 | Train Acc: 62.8% | Val Acc: 54.3%
Epoch [11/20] | Loss: 0.9143 | Train Acc: 60.2% | Val Acc: 50.0%
Epoch [12/20] | Loss: 0.8433 | Train Acc: 63.6% | Val Acc: 54.3%
Epoch [13/20] | Loss: 0.8303 | Train Acc: 64.4% | Val Acc: 53.2%
Epoch [14/20] | Loss

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#AUGMENTATION & STANDARDIZATION LOADER
class AugmentedSkeletonDataset(Dataset):
    def __init__(self, metadata, target_frames=60, target_joints=33, is_training=False):
        self.metadata = metadata
        self.T = target_frames
        self.V = target_joints
        self.is_training = is_training

    def __len__(self):
        return len(self.metadata)

    def augment_skeleton(self, data):
        # 1. Random Spatial Jitter (Simulates slight camera shifting)
        if np.random.rand() > 0.5:
            noise = np.random.normal(0, 0.05, data.shape).astype(np.float32)
            data += noise

        # 2. Random Temporal Crop/Shift (Starts the action at a slightly different frame)
        if np.random.rand() > 0.5 and data.shape[1] > 10:
            shift = np.random.randint(1, 10)
            data = np.roll(data, shift, axis=1)

        return data

    def __getitem__(self, idx):
        item = self.metadata[idx]
        data = np.load(item["file_path"]).astype(np.float32)

        # Handle dimensions (2D, 3D, 4D standardizer from before)
        data = np.squeeze(data)
        if data.ndim == 2:
            t_len, feat_len = data.shape
            if feat_len % 3 == 0:
                data = data.reshape(t_len, feat_len // 3, 3)
            else:
                data = data.reshape(t_len, feat_len // 2, 2)
        if data.ndim == 3:
            if data.shape[-1] in [2, 3]: data = np.transpose(data, (2, 0, 1))
            elif data.shape[1] in [2, 3]: data = np.transpose(data, (1, 0, 2))
            data = np.expand_dims(data, axis=-1)
        elif data.ndim == 4:
            if data.shape[0] not in [2, 3] and data.shape[-1] in [2, 3]:
                data = np.transpose(data, (3, 0, 1, 2))
        while data.ndim < 4: data = np.expand_dims(data, axis=-1)

        # Apply Augmentation only during training
        if self.is_training:
            data = self.augment_skeleton(data)

        # Standardize Output Shape (3, T, V, 1)
        out_data = np.zeros((3, self.T, self.V, 1), dtype=np.float32)
        c_limit, t_limit, v_limit = min(data.shape[0], 3), min(data.shape[1], self.T), min(data.shape[2], self.V)
        out_data[:c_limit, :t_limit, :v_limit, 0] = data[:c_limit, :t_limit, :v_limit, 0]

        tensor_data = torch.from_numpy(out_data).float()
        label = torch.tensor(item["label"], dtype=torch.long)
        return tensor_data, label

# --- 2. Building Loaders ---
INDEX_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset/master_balanced_4class_index.json"
with open(INDEX_PATH, "r") as f:
    balanced_meta = json.load(f)

labels = [x["label"] for x in balanced_meta]
train_meta, val_meta = train_test_split(balanced_meta, test_size=0.2, random_state=42, stratify=labels)

# Enable augmentation on train_loader only
train_loader = DataLoader(AugmentedSkeletonDataset(train_meta, is_training=True), batch_size=16, shuffle=True)
val_loader = DataLoader(AugmentedSkeletonDataset(val_meta, is_training=False), batch_size=16, shuffle=False)

# --- 3. DEEPER ST-GCN ARCHITECTURE ---
class DeeperSTGCN(nn.Module):
    def __init__(self, num_classes=4):
        super(DeeperSTGCN, self).__init__()

        self.conv1 = nn.Conv2d(3, 64, kernel_size=(5, 1), padding=(2, 0))
        self.bn1 = nn.BatchNorm2d(64)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=(5, 1), stride=(2, 1), padding=(2, 0))
        self.bn2 = nn.BatchNorm2d(128)

        # Added a 3rd deeper block to catch finer behavioral details
        self.conv3 = nn.Conv2d(128, 256, kernel_size=(3, 1), stride=(2, 1), padding=(1, 0))
        self.bn3 = nn.BatchNorm2d(256)

        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.5) # Increased dropout to prevent overfitting
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.view(N, C, T, V * M)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))

        x = self.pool(x)
        x = x.view(N, -1)
        x = self.dropout(x)
        return self.fc(x)

# --- 4. Training Loop with Learning Rate Scheduler ---
model = DeeperSTGCN(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Gradually lower the learning rate if the model gets stuck
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

EPOCHS = 30 # Increased epochs because augmentation takes longer to learn
best_val_acc = 0.0
MODEL_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset/advanced_stgcn_4class.pth"

print(f"Starting Augmented Training on {device}...")

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")

    # Step the scheduler based on validation accuracy
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

print(f"\nAdvanced Training Complete! Best Val Accuracy: {best_val_acc:.1f}%")

Starting Augmented Training on cuda...
Epoch [01/30] | Loss: 1.0442 | Train Acc: 49.7% | Val Acc: 47.9%
Epoch [02/30] | Loss: 1.0039 | Train Acc: 50.8% | Val Acc: 47.9%
Epoch [03/30] | Loss: 0.9838 | Train Acc: 56.1% | Val Acc: 44.7%
Epoch [04/30] | Loss: 0.9053 | Train Acc: 59.9% | Val Acc: 44.7%
Epoch [05/30] | Loss: 0.8244 | Train Acc: 64.4% | Val Acc: 45.7%
Epoch [06/30] | Loss: 0.8052 | Train Acc: 65.8% | Val Acc: 44.7%
Epoch [07/30] | Loss: 0.8673 | Train Acc: 57.2% | Val Acc: 52.1%
Epoch [08/30] | Loss: 0.8124 | Train Acc: 67.6% | Val Acc: 46.8%
Epoch [09/30] | Loss: 0.7797 | Train Acc: 66.8% | Val Acc: 46.8%
Epoch [10/30] | Loss: 0.7739 | Train Acc: 67.9% | Val Acc: 44.7%
Epoch [11/30] | Loss: 0.7571 | Train Acc: 68.4% | Val Acc: 45.7%
Epoch [12/30] | Loss: 0.8115 | Train Acc: 70.3% | Val Acc: 55.3%
Epoch [13/30] | Loss: 0.7521 | Train Acc: 71.1% | Val Acc: 45.7%
Epoch [14/30] | Loss: 0.7398 | Train Acc: 69.3% | Val Acc: 50.0%
Epoch [15/30] | Loss: 0.6963 | Train Acc: 71.9% | V

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. LOADER (Same as before) ---
class AugmentedSkeletonDataset(Dataset):
    def __init__(self, metadata, target_frames=60, target_joints=33, is_training=False):
        self.metadata = metadata
        self.T = target_frames
        self.V = target_joints
        self.is_training = is_training

    def __len__(self):
        return len(self.metadata)

    def augment_skeleton(self, data):
        if np.random.rand() > 0.5:
            data += np.random.normal(0, 0.05, data.shape).astype(np.float32)
        if np.random.rand() > 0.5 and data.shape[1] > 10:
            data = np.roll(data, np.random.randint(1, 10), axis=1)
        return data

    def __getitem__(self, idx):
        item = self.metadata[idx]
        data = np.load(item["file_path"]).astype(np.float32)

        data = np.squeeze(data)
        if data.ndim == 2:
            t_len, feat_len = data.shape
            data = data.reshape(t_len, feat_len // (3 if feat_len % 3 == 0 else 2), (3 if feat_len % 3 == 0 else 2))
        if data.ndim == 3:
            if data.shape[-1] in [2, 3]: data = np.transpose(data, (2, 0, 1))
            elif data.shape[1] in [2, 3]: data = np.transpose(data, (1, 0, 2))
            data = np.expand_dims(data, axis=-1)
        elif data.ndim == 4:
            if data.shape[0] not in [2, 3] and data.shape[-1] in [2, 3]:
                data = np.transpose(data, (3, 0, 1, 2))
        while data.ndim < 4: data = np.expand_dims(data, axis=-1)

        if self.is_training: data = self.augment_skeleton(data)

        out_data = np.zeros((3, self.T, self.V, 1), dtype=np.float32)
        c_limit, t_limit, v_limit = min(data.shape[0], 3), min(data.shape[1], self.T), min(data.shape[2], self.V)
        out_data[:c_limit, :t_limit, :v_limit, 0] = data[:c_limit, :t_limit, :v_limit, 0]

        return torch.from_numpy(out_data).float(), torch.tensor(item["label"], dtype=torch.long)

# --- 2. LOADERS ---
INDEX_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset/master_balanced_4class_index.json"
with open(INDEX_PATH, "r") as f:
    balanced_meta = json.load(f)

labels = [x["label"] for x in balanced_meta]
train_meta, val_meta = train_test_split(balanced_meta, test_size=0.2, random_state=42, stratify=labels)

train_loader = DataLoader(AugmentedSkeletonDataset(train_meta, is_training=True), batch_size=16, shuffle=True)
val_loader = DataLoader(AugmentedSkeletonDataset(val_meta, is_training=False), batch_size=16, shuffle=False)

# --- 3. HIGH-REGULARIZATION ST-GCN ---
class OptimizedSTGCN(nn.Module):
    def __init__(self, num_classes=4):
        super(OptimizedSTGCN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=(5, 1), padding=(2, 0))
        self.bn1 = nn.BatchNorm2d(64)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=(5, 1), stride=(2, 1), padding=(2, 0))
        self.bn2 = nn.BatchNorm2d(128)

        self.conv3 = nn.Conv2d(128, 256, kernel_size=(3, 1), stride=(2, 1), padding=(1, 0))
        self.bn3 = nn.BatchNorm2d(256)

        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # Increased Dropout from 0.3 to 0.6 to brutally force generalization
        self.dropout = nn.Dropout(0.6)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.view(N, C, T, V * M)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = x.view(N, -1)
        x = self.dropout(x)
        return self.fc(x)

# --- 4. TRAINING LOOP WITH EARLY STOPPING ---
model = OptimizedSTGCN(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()

# Increased Weight Decay (L2 Regularization) to penalize memorization
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

EPOCHS = 40
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 7
MODEL_PATH = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset/optimized_stgcn_4class.pth"

print(f"Starting Optimized Training on {device}...")

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}% | LR: {current_lr}")

    scheduler.step(avg_val_loss)

    # Save the model based on the lowest Validation Loss, not just accuracy
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered! Model stopped improving for {EARLY_STOP_PATIENCE} epochs.")
        break

print("-" * 60)
print(f"Optimized Training Complete! Best Val Loss: {best_val_loss:.4f}")
print(f"Saved optimal model to: {MODEL_PATH}")

Starting Optimized Training on cuda...
Epoch [01/40] | Train Loss: 1.0987 | Val Loss: 1.0297 | Val Acc: 47.9% | LR: 0.001
Epoch [02/40] | Train Loss: 1.0013 | Val Loss: 1.0233 | Val Acc: 48.9% | LR: 0.001
Epoch [03/40] | Train Loss: 0.9656 | Val Loss: 1.0220 | Val Acc: 47.9% | LR: 0.001
Epoch [04/40] | Train Loss: 0.9864 | Val Loss: 1.0853 | Val Acc: 45.7% | LR: 0.001
Epoch [05/40] | Train Loss: 0.9030 | Val Loss: 1.1350 | Val Acc: 44.7% | LR: 0.001
Epoch [06/40] | Train Loss: 0.8527 | Val Loss: 0.9757 | Val Acc: 44.7% | LR: 0.001
Epoch [07/40] | Train Loss: 0.9476 | Val Loss: 1.0249 | Val Acc: 53.2% | LR: 0.001
Epoch [08/40] | Train Loss: 0.8045 | Val Loss: 0.9536 | Val Acc: 45.7% | LR: 0.001
Epoch [09/40] | Train Loss: 0.8128 | Val Loss: 0.9406 | Val Acc: 58.5% | LR: 0.001
Epoch [10/40] | Train Loss: 0.8442 | Val Loss: 1.0293 | Val Acc: 48.9% | LR: 0.001
Epoch [11/40] | Train Loss: 0.7780 | Val Loss: 0.9333 | Val Acc: 46.8% | LR: 0.001
Epoch [12/40] | Train Loss: 0.8224 | Val Loss: 1

**Class Balancing**

In [ ]:
import os
import json
import glob
import random
import numpy as np
from tqdm import tqdm

WORKING_DIR = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped"
OUTPUT_DIR  = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset"
SYNTH_DIR   = os.path.join(OUTPUT_DIR, "Synthetic_Data")
os.makedirs(SYNTH_DIR, exist_ok=True)

TARGET_SAMPLES = 2500

#Augmentation Functions
def generate_synthetic_skeleton(data, aug_type):
    synth_data = data.copy().astype(np.float32)

    if aug_type == "noise":
        # Adding slight random noise to coordinates
        noise = np.random.normal(0, 0.02, synth_data.shape).astype(np.float32)
        synth_data += noise

    elif aug_type == "shift":
        # Shifting frames temporally (wrap around)
        if synth_data.ndim >= 2 and synth_data.shape[1] > 5:
            shift = random.randint(2, synth_data.shape[1] // 2)
            synth_data = np.roll(synth_data, shift, axis=1)

    elif aug_type == "scale":
        # Simulating taller/shorter subjects (scale X,Y,Z by 90% to 110%)
        scale_factor = random.uniform(0.9, 1.1)
        synth_data *= scale_factor

    return synth_data

#Collecting Original Raw Files
raw_classes = {0: [], 1: [], 2: [], 3: []}

# Unpacked HBD21 (Class 3 & 0)
for f in glob.glob(os.path.join(OUTPUT_DIR, "hbd21_unpacked/*.npy")):
    label = 0 if "normal" in f.lower() else 3
    pass
all_npy_files = glob.glob(os.path.join(WORKING_DIR, "**", "*.npy"), recursive=True)
all_npy_files += glob.glob(os.path.join(OUTPUT_DIR, "hbd21_unpacked/*.npy"))

for f in all_npy_files:
    f_lower = f.lower()
    if "asdpose" in f_lower or "ssbd" in f_lower:
        raw_classes[1].append(f)
    elif "ur_fall" in f_lower or "le2i" in f_lower:
        if "not fall" in f_lower or "adl" in f_lower or "normal" in f_lower:
            raw_classes[0].append(f)
        elif "fall" in f_lower:
            raw_classes[2].append(f)
        else:
            raw_classes[2].append(f) # Le2i default
    elif "hbd21_unpacked" in f_lower:
        pass

**Retraining Model with class-balancing**

In [ ]:
import os
import json
import glob
import random
import numpy as np
from tqdm import tqdm

WORKING_DIR = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/skeletons_unzipped"
OUTPUT_DIR  = "/content/drive/MyDrive/Capstone_Project/Datasets_Usage/ProcessedSkeletonDataset"
SYNTH_DIR   = os.path.join(OUTPUT_DIR, "Synthetic_Data")
os.makedirs(SYNTH_DIR, exist_ok=True)

TARGET_SAMPLES = 2500

#Loading the pristine Phase 1 balanced index to get our guaranteed clean base files
OLD_INDEX_PATH = os.path.join(OUTPUT_DIR, "master_balanced_4class_index.json")
with open(OLD_INDEX_PATH, "r") as f:
    base_data = json.load(f)

# Grouping the clean base files by class
class_bases = {0: [], 1: [], 2: [], 3: []}
for item in base_data:
    class_bases[item["label"]].append(item["file_path"])

# 2. Replacing Class 1 (Stimming) with 2,500 raw samples from ASDPose
print("Gathering massive Stimming dataset...")
all_asd_files = glob.glob(os.path.join(WORKING_DIR, "asdpose/**/*.npy"), recursive=True)
class_bases[1] = random.sample(all_asd_files, TARGET_SAMPLES)

# 3. Augmentation Pipeline
def augment_and_save(source_path, save_path, aug_type):
    data = np.load(source_path).astype(np.float32)

    if aug_type == "noise":
        data += np.random.normal(0, 0.02, data.shape).astype(np.float32)
    elif aug_type == "shift" and data.ndim >= 2 and data.shape[1] > 5:
        data = np.roll(data, random.randint(2, data.shape[1] // 2), axis=1)
    elif aug_type == "scale":
        data *= random.uniform(0.9, 1.1)

    np.save(save_path, data)
    return save_path

#Upscaling Minority Classes (0, 2, 3)
massive_index = []
aug_types = ["noise", "shift", "scale"]

for label in [0, 1, 2, 3]:
    print(f"\nProcessing Class {label}...")
    current_files = class_bases[label]

    # Adding all original files to the new index
    for f in current_files:
        massive_index.append({"file_path": f, "label": label})

    # If we need more files to hit 2,500, generate them synthetically
    if len(current_files) < TARGET_SAMPLES:
        shortfall = TARGET_SAMPLES - len(current_files)
        print(f"Generating {shortfall} synthetic samples...")

        for i in tqdm(range(shortfall)):
            # Pick a random original file to mutate
            source_file = random.choice(current_files)
            aug_type = random.choice(aug_types)

            synth_filename = f"synth_c{label}_{i}_{aug_type}.npy"
            synth_path = os.path.join(SYNTH_DIR, synth_filename)

            new_path = augment_and_save(source_file, synth_path, aug_type)
            massive_index.append({"file_path": new_path, "label": label})

# 5. Shuffle and Save
random.shuffle(massive_index)
MASSIVE_INDEX_PATH = os.path.join(OUTPUT_DIR, "massive_10k_dataset_index.json")

with open(MASSIVE_INDEX_PATH, "w") as f:
    json.dump(massive_index, f, indent=4)

print(f"\nSynthetic Upscaling Complete! Index saved to {MASSIVE_INDEX_PATH}")
print(f"Total Model-Ready Videos: {len(massive_index)}")